# Membership Inference Attacks on DocVQA Generative Models

Notebook for Google Colab A100. It evaluates membership inference before and after fine-tuning for Florence-2 and Qwen2-VL models, downloads checkpoints from Hugging Face Hub, and logs metrics, plots, and CSV artifacts to Comet.


## 0. Setup

In [ ]:
%pip install -q huggingface_hub comet_ml python-dotenv scipy scikit-learn matplotlib peft bitsandbytes

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

from huggingface_hub import snapshot_download


def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists() and (candidate / 'requirements.txt').exists():
            return candidate
    return None


def get_colab_secret(name: str) -> str | None:
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.getenv(name)


project_root = find_project_root(Path.cwd())
repo_url = get_colab_secret('COURSE_WORK2026_REPO_URL')

if project_root is None:
    clone_dir = Path('/content/course_work2026')
    if not (clone_dir / '.git').exists():
        if not repo_url:
            raise RuntimeError('Could not detect the repository locally and COURSE_WORK2026_REPO_URL is not set in Colab secrets.')
        subprocess.run(['git', 'clone', repo_url, str(clone_dir)], check=True)
    project_root = clone_dir

sys.path.insert(0, str(project_root / 'src'))

from colab_setup import setup_colab

setup_summary, bootstrap_experiment = setup_colab(repo_url=repo_url)
bootstrap_experiment.set_name('mia-bootstrap')
bootstrap_experiment.end()

PROJECT_ROOT = Path(setup_summary['project_root'])
ARTIFACTS_ROOT = PROJECT_ROOT / 'artifacts'
MIA_OUTPUT_DIR = ARTIFACTS_ROOT / 'privacy_attacks' / 'mia'
CHECKPOINTS_ROOT = PROJECT_ROOT / 'checkpoints'
HF_CHECKPOINT_REPO = os.getenv('HF_MODEL_REPO') or 'karimkramin/docvqa-privacy-checkpoints'

MIA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_ROOT.mkdir(parents=True, exist_ok=True)

snapshot_path = snapshot_download(repo_id=HF_CHECKPOINT_REPO, repo_type='model', local_dir=str(CHECKPOINTS_ROOT), token=os.getenv('HF_TOKEN'))

print(json.dumps(setup_summary, ensure_ascii=False, indent=2))
print({'project_root': str(PROJECT_ROOT), 'checkpoints_root': str(CHECKPOINTS_ROOT), 'snapshot_path': str(snapshot_path), 'mia_output_dir': str(MIA_OUTPUT_DIR)})

## 1. MIA Utilities

In [ ]:
from typing import Any

import comet_ml
import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from peft import PeftModel
from PIL import Image
from scipy.stats import ttest_ind
from sklearn.metrics import roc_auc_score
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig, Qwen2VLForConditionalGeneration

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32


def load_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def resolve_image_path(raw_path: str, split: str) -> Path:
    original = Path(raw_path)
    if original.exists():
        return original
    benchmark_dir = 'benchmark_train' if split == 'train' else 'benchmark'
    candidate = ARTIFACTS_ROOT / 'docqa_recovery' / benchmark_dir / 'images' / 'original' / original.name
    if candidate.exists():
        return candidate
    raise FileNotFoundError(f'Cannot resolve image path for {raw_path}')


def load_mia_records(manifest_path: Path, limit: int | None = None) -> list[dict[str, Any]]:
    records = []
    for row in load_jsonl(manifest_path):
        records.append({
            'question_id': row['example_id'],
            'example_id': row['example_id'],
            'split': row['split'],
            'question': row['question'],
            'gold_answer': row['answer'],
            'coarse_type': row['coarse_field_type'],
            'image_path': str(resolve_image_path(str(row['image_path']), str(row['split']))),
        })
    return records[:limit] if limit is not None else records


TRAIN_RECORDS = load_mia_records(ARTIFACTS_ROOT / 'docqa_recovery' / 'benchmark_train' / 'manifest.jsonl', limit=800)
VALIDATION_RECORDS = load_mia_records(ARTIFACTS_ROOT / 'docqa_recovery' / 'benchmark' / 'manifest.jsonl', limit=800)


def qwen_user_text(question: str) -> str:
    return f'Answer the document question with a short span copied from the document when possible.\nQuestion: {question}'


def mark_model_kind(processor, kind: str) -> None:
    setattr(processor, '_mia_model_kind', kind)


def get_model_kind(model, processor) -> str:
    kind = getattr(processor, '_mia_model_kind', None)
    if kind:
        return kind
    model_type = getattr(getattr(model, 'config', None), 'model_type', '')
    if 'qwen2_vl' in model_type or 'qwen2-vl' in model_type:
        return 'qwen2vl'
    return 'florence'


def prepare_generation_inputs(model, processor, image: Image.Image, question: str) -> tuple[dict[str, torch.Tensor], int]:
    kind = get_model_kind(model, processor)
    if kind == 'florence':
        prompt = f'<VQA>{question}'
        inputs = processor(images=image, text=prompt, return_tensors='pt')
        inputs = {k: v.to(device if isinstance(v, torch.Tensor) else v) for k, v in inputs.items()}
        if 'pixel_values' in inputs:
            inputs['pixel_values'] = inputs['pixel_values'].to(device=device, dtype=torch_dtype)
        return inputs, int(inputs['input_ids'].shape[1])

    messages = [{'role': 'user', 'content': [{'type': 'image', 'image': 'placeholder'}, {'type': 'text', 'text': qwen_user_text(question)}]}]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt], images=[image], padding=True, return_tensors='pt')
    inputs = {k: v.to(model.device if isinstance(v, torch.Tensor) else v) for k, v in inputs.items()}
    return inputs, int(inputs['input_ids'].shape[1])


def load_florence(checkpoint_path: Path | None = None):
    source = str(checkpoint_path) if checkpoint_path else 'microsoft/Florence-2-base'
    processor = AutoProcessor.from_pretrained(source, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(source, torch_dtype=torch_dtype, trust_remote_code=True).to(device)
    model.eval()
    mark_model_kind(processor, 'florence')
    return model, processor


def load_qwen(base_model_name: str, adapter_path: Path | None = None):
    quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
    processor = AutoProcessor.from_pretrained(str(adapter_path) if adapter_path else base_model_name, trust_remote_code=True)
    model = Qwen2VLForConditionalGeneration.from_pretrained(base_model_name, quantization_config=quantization_config, device_map='auto', trust_remote_code=True)
    if adapter_path is not None:
        model = PeftModel.from_pretrained(model, adapter_path)
    model.eval()
    mark_model_kind(processor, 'qwen2vl')
    return model, processor


In [ ]:
def compute_generation_confidence(model, processor, image, question) -> float:
    image = image.convert('RGB') if isinstance(image, Image.Image) else Image.open(image).convert('RGB')
    inputs, prompt_len = prepare_generation_inputs(model, processor, image, question)
    with torch.no_grad():
        generated = model.generate(**inputs, max_new_tokens=50, do_sample=False, return_dict_in_generate=True, output_scores=True)
    if not generated.scores:
        return float('nan')
    generated_ids = generated.sequences[:, prompt_len:]
    step_count = min(generated_ids.shape[1], len(generated.scores))
    if step_count == 0:
        return float('nan')
    token_log_probs = []
    for step in range(step_count):
        step_log_probs = torch.log_softmax(generated.scores[step], dim=-1)
        token_id = generated_ids[0, step].item()
        token_log_probs.append(float(step_log_probs[0, token_id].item()))
    return float(sum(token_log_probs) / len(token_log_probs))


def compute_loss_on_gold(model, processor, image, question, gold_answer) -> float:
    image = image.convert('RGB') if isinstance(image, Image.Image) else Image.open(image).convert('RGB')
    kind = get_model_kind(model, processor)
    if kind == 'florence':
        prompt = f'<VQA>{question}'
        inputs = processor(images=image, text=prompt, return_tensors='pt')
        labels = processor.tokenizer(gold_answer, return_tensors='pt', add_special_tokens=True).input_ids
        inputs = {k: v.to(device if isinstance(v, torch.Tensor) else v) for k, v in inputs.items()}
        if 'pixel_values' in inputs:
            inputs['pixel_values'] = inputs['pixel_values'].to(device=device, dtype=torch_dtype)
        labels = labels.to(device)
        with torch.no_grad():
            outputs = model(**inputs, labels=labels)
        return float(outputs.loss.item())

    prompt_messages = [{'role': 'user', 'content': [{'type': 'image', 'image': 'placeholder'}, {'type': 'text', 'text': qwen_user_text(question)}]}]
    full_messages = prompt_messages + [{'role': 'assistant', 'content': [{'type': 'text', 'text': gold_answer}]}]
    prompt_text = processor.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    full_text = processor.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)
    prompt_batch = processor(text=[prompt_text], images=[image], padding=True, return_tensors='pt')
    full_batch = processor(text=[full_text], images=[image], padding=True, return_tensors='pt')
    prompt_batch = {k: v.to(model.device if isinstance(v, torch.Tensor) else v) for k, v in prompt_batch.items()}
    full_batch = {k: v.to(model.device if isinstance(v, torch.Tensor) else v) for k, v in full_batch.items()}
    labels = full_batch['input_ids'].clone()
    labels[full_batch['attention_mask'] == 0] = -100
    prompt_len = int(prompt_batch['attention_mask'].sum().item())
    labels[:, :prompt_len] = -100
    with torch.no_grad():
        outputs = model(input_ids=full_batch['input_ids'], attention_mask=full_batch['attention_mask'], pixel_values=full_batch.get('pixel_values'), image_grid_thw=full_batch.get('image_grid_thw'), labels=labels)
    return float(outputs.loss.item())


def run_mia(model, processor, seen_records, unseen_records) -> pd.DataFrame:
    rows = []
    model.eval()
    for is_seen, records in [(1, seen_records), (0, unseen_records)]:
        for record in records:
            image = Image.open(record['image_path']).convert('RGB')
            confidence = compute_generation_confidence(model, processor, image, record['question'])
            loss = compute_loss_on_gold(model, processor, image, record['question'], record['gold_answer'])
            rows.append({'question_id': record['question_id'], 'example_id': record['example_id'], 'is_seen': is_seen, 'split': record['split'], 'coarse_type': record['coarse_type'], 'question': record['question'], 'gold_answer': record['gold_answer'], 'confidence': confidence, 'loss': loss})
    return pd.DataFrame(rows)


def compute_auc_safe(labels, scores) -> float:
    labels = list(labels)
    scores = list(scores)
    if len(set(labels)) < 2:
        return float('nan')
    return float(roc_auc_score(labels, scores))


def summarize_mia(df: pd.DataFrame) -> dict[str, float]:
    seen_conf = df.loc[df['is_seen'] == 1, 'confidence']
    unseen_conf = df.loc[df['is_seen'] == 0, 'confidence']
    seen_loss = df.loc[df['is_seen'] == 1, 'loss']
    unseen_loss = df.loc[df['is_seen'] == 0, 'loss']
    return {
        'auc_confidence': compute_auc_safe(df['is_seen'], df['confidence']),
        'auc_loss': compute_auc_safe(df['is_seen'], -df['loss']),
        'pvalue_confidence': float(ttest_ind(seen_conf, unseen_conf, equal_var=False, nan_policy='omit').pvalue),
        'pvalue_loss': float(ttest_ind(seen_loss, unseen_loss, equal_var=False, nan_policy='omit').pvalue),
        'mean_confidence_seen': float(seen_conf.mean()),
        'std_confidence_seen': float(seen_conf.std(ddof=0)),
        'mean_confidence_unseen': float(unseen_conf.mean()),
        'std_confidence_unseen': float(unseen_conf.std(ddof=0)),
        'mean_loss_seen': float(seen_loss.mean()),
        'std_loss_seen': float(seen_loss.std(ddof=0)),
        'mean_loss_unseen': float(unseen_loss.mean()),
        'std_loss_unseen': float(unseen_loss.std(ddof=0)),
    }


def coarse_type_auc_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for coarse_type, group in df.groupby('coarse_type'):
        if group['is_seen'].nunique() < 2:
            continue
        rows.append({'coarse_type': coarse_type, 'n_examples': int(len(group)), 'auc_confidence': compute_auc_safe(group['is_seen'], group['confidence']), 'auc_loss': compute_auc_safe(group['is_seen'], -group['loss'])})
    return pd.DataFrame(rows).sort_values('coarse_type').reset_index(drop=True)


def make_histograms(df: pd.DataFrame, title_prefix: str):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(df.loc[df['is_seen'] == 1, 'confidence'], bins=30, alpha=0.6, label='seen')
    axes[0].hist(df.loc[df['is_seen'] == 0, 'confidence'], bins=30, alpha=0.6, label='unseen')
    axes[0].set_title(f'{title_prefix}: confidence')
    axes[0].legend()
    axes[1].hist(df.loc[df['is_seen'] == 1, 'loss'], bins=30, alpha=0.6, label='seen')
    axes[1].hist(df.loc[df['is_seen'] == 0, 'loss'], bins=30, alpha=0.6, label='unseen')
    axes[1].set_title(f'{title_prefix}: loss')
    axes[1].legend()
    plt.tight_layout()
    return fig


def log_csv_asset(experiment, csv_path: Path) -> None:
    experiment.log_asset(str(csv_path), file_name=csv_path.name)


BASELINE_SPECS = [
    {'tag': 'florence2', 'display_name': 'Florence-2-base', 'kind': 'florence', 'base_model_name': 'microsoft/Florence-2-base'},
    {'tag': 'qwen2b', 'display_name': 'Qwen2-VL-2B-Instruct', 'kind': 'qwen2vl', 'base_model_name': 'Qwen/Qwen2-VL-2B-Instruct'},
    {'tag': 'qwen7b', 'display_name': 'Qwen2-VL-7B-Instruct', 'kind': 'qwen2vl', 'base_model_name': 'Qwen/Qwen2-VL-7B-Instruct'},
]
FINETUNED_SPECS = [
    {'tag': 'florence2', 'display_name': 'Florence-2-base', 'kind': 'florence', 'base_model_name': 'microsoft/Florence-2-base', 'checkpoint_path': CHECKPOINTS_ROOT / 'florence2' / 'epoch_50', 'epochs': [1, 3, 10, 30, 50]},
    {'tag': 'qwen2b', 'display_name': 'Qwen2-VL-2B-Instruct', 'kind': 'qwen2vl', 'base_model_name': 'Qwen/Qwen2-VL-2B-Instruct', 'checkpoint_path': CHECKPOINTS_ROOT / 'qwen2b' / 'epoch_30', 'epochs': [1, 3, 10, 30]},
    {'tag': 'qwen7b', 'display_name': 'Qwen2-VL-7B-Instruct', 'kind': 'qwen2vl', 'base_model_name': 'Qwen/Qwen2-VL-7B-Instruct', 'checkpoint_path': CHECKPOINTS_ROOT / 'qwen7b' / 'epoch_30', 'epochs': [1, 3, 10, 30]},
]
print({'train_records': len(TRAIN_RECORDS), 'validation_records': len(VALIDATION_RECORDS)})

## 2. Baseline MIA Before Fine-tuning

In [ ]:
baseline_summary_rows = []

for spec in BASELINE_SPECS:
    experiment = comet_ml.Experiment(project_name='docvqa-privacy')
    experiment.set_name(f"{spec['tag']}-mia-baseline")
    experiment.log_parameters({'stage': 'baseline_mia', 'model_tag': spec['tag'], 'model_name': spec['base_model_name'], 'weights_source': 'base_pretrained_model', 'adapter_path': '', 'n_seen': 800, 'n_unseen': 800})

    if spec['kind'] == 'florence':
        model, processor = load_florence()
    else:
        model, processor = load_qwen(spec['base_model_name'])

    mia_df = run_mia(model, processor, TRAIN_RECORDS, VALIDATION_RECORDS)
    summary = summarize_mia(mia_df)
    summary['tag'] = spec['tag']
    summary['stage'] = 'baseline'
    baseline_summary_rows.append(summary)

    csv_path = MIA_OUTPUT_DIR / f"{spec['tag']}_baseline_mia_scores.csv"
    mia_df.to_csv(csv_path, index=False)
    experiment.log_metric('auc_confidence', summary['auc_confidence'])
    experiment.log_metric('auc_loss', summary['auc_loss'])
    experiment.log_metric('mean_confidence_seen', summary['mean_confidence_seen'])
    experiment.log_metric('std_confidence_seen', summary['std_confidence_seen'])
    experiment.log_metric('mean_confidence_unseen', summary['mean_confidence_unseen'])
    experiment.log_metric('std_confidence_unseen', summary['std_confidence_unseen'])
    experiment.log_metric('mean_loss_seen', summary['mean_loss_seen'])
    experiment.log_metric('std_loss_seen', summary['std_loss_seen'])
    experiment.log_metric('mean_loss_unseen', summary['mean_loss_unseen'])
    experiment.log_metric('std_loss_unseen', summary['std_loss_unseen'])
    experiment.log_table('baseline_mia_scores.csv', mia_df)
    log_csv_asset(experiment, csv_path)
    print({'tag': spec['tag'], 'auc_confidence': summary['auc_confidence'], 'auc_loss': summary['auc_loss'], 'expected_baseline': 'AUC ~= 0.5'})
    experiment.end()
    del model
    torch.cuda.empty_cache()

baseline_summary_df = pd.DataFrame(baseline_summary_rows)
baseline_summary_csv = MIA_OUTPUT_DIR / 'baseline_mia_summary.csv'
baseline_summary_df.to_csv(baseline_summary_csv, index=False)
baseline_summary_experiment = comet_ml.Experiment(project_name='docvqa-privacy')
baseline_summary_experiment.set_name('mia-baseline-summary')
baseline_summary_experiment.log_table('baseline_mia_summary.csv', baseline_summary_df)
log_csv_asset(baseline_summary_experiment, baseline_summary_csv)
baseline_summary_experiment.end()
display(baseline_summary_df)

## 3. MIA After Fine-tuning

In [ ]:
finetuned_summary_rows = []

for spec in FINETUNED_SPECS:
    if not spec['checkpoint_path'].exists():
        raise FileNotFoundError(f"Checkpoint not found: {spec['checkpoint_path']}")

    experiment = comet_ml.Experiment(project_name='docvqa-privacy')
    experiment.set_name(f"{spec['tag']}-mia-finetuned")
    experiment.log_parameters({'stage': 'finetuned_mia', 'model_tag': spec['tag'], 'model_name': spec['base_model_name'], 'weights_source': 'finetuned_checkpoint', 'checkpoint_path': str(spec['checkpoint_path']), 'adapter_path': str(spec['checkpoint_path']) if spec['kind'] == 'qwen2vl' else '', 'n_seen': 800, 'n_unseen': 800})

    if spec['kind'] == 'florence':
        model, processor = load_florence(spec['checkpoint_path'])
    else:
        model, processor = load_qwen(spec['base_model_name'], spec['checkpoint_path'])

    mia_df = run_mia(model, processor, TRAIN_RECORDS, VALIDATION_RECORDS)
    summary = summarize_mia(mia_df)
    summary['tag'] = spec['tag']
    summary['stage'] = 'finetuned'
    finetuned_summary_rows.append(summary)

    coarse_auc_df = coarse_type_auc_table(mia_df)
    scores_csv = MIA_OUTPUT_DIR / f"{spec['tag']}_mia_scores.csv"
    coarse_csv = MIA_OUTPUT_DIR / f"{spec['tag']}_mia_coarse_auc.csv"
    mia_df.to_csv(scores_csv, index=False)
    coarse_auc_df.to_csv(coarse_csv, index=False)
    fig = make_histograms(mia_df, title_prefix=spec['tag'])

    experiment.log_metric('auc_confidence', summary['auc_confidence'])
    experiment.log_metric('auc_loss', summary['auc_loss'])
    experiment.log_metric('pvalue_confidence', summary['pvalue_confidence'])
    experiment.log_metric('pvalue_loss', summary['pvalue_loss'])
    experiment.log_metric('mean_confidence_seen', summary['mean_confidence_seen'])
    experiment.log_metric('std_confidence_seen', summary['std_confidence_seen'])
    experiment.log_metric('mean_confidence_unseen', summary['mean_confidence_unseen'])
    experiment.log_metric('std_confidence_unseen', summary['std_confidence_unseen'])
    experiment.log_metric('mean_loss_seen', summary['mean_loss_seen'])
    experiment.log_metric('std_loss_seen', summary['std_loss_seen'])
    experiment.log_metric('mean_loss_unseen', summary['mean_loss_unseen'])
    experiment.log_metric('std_loss_unseen', summary['std_loss_unseen'])
    experiment.log_figure('confidence_histogram', fig)
    experiment.log_table('mia_scores.csv', mia_df)
    experiment.log_table('mia_coarse_auc.csv', coarse_auc_df)
    log_csv_asset(experiment, scores_csv)
    log_csv_asset(experiment, coarse_csv)
    display(coarse_auc_df)
    print({'tag': spec['tag'], **summary})
    experiment.end()
    plt.close(fig)
    del model
    torch.cuda.empty_cache()

finetuned_summary_df = pd.DataFrame(finetuned_summary_rows)
finetuned_summary_csv = MIA_OUTPUT_DIR / 'finetuned_mia_summary.csv'
finetuned_summary_df.to_csv(finetuned_summary_csv, index=False)
finetuned_summary_experiment = comet_ml.Experiment(project_name='docvqa-privacy')
finetuned_summary_experiment.set_name('mia-finetuned-summary')
finetuned_summary_experiment.log_table('finetuned_mia_summary.csv', finetuned_summary_df)
log_csv_asset(finetuned_summary_experiment, finetuned_summary_csv)
finetuned_summary_experiment.end()
display(finetuned_summary_df)

## 4. Epoch Curve

In [ ]:
epoch_curve_rows = []
seen_small = TRAIN_RECORDS[:100]
unseen_small = VALIDATION_RECORDS[:100]

experiment = comet_ml.Experiment(project_name='docvqa-privacy')
experiment.set_name('mia-epoch-curve')
experiment.log_parameters({'stage': 'epoch_curve', 'n_seen': len(seen_small), 'n_unseen': len(unseen_small)})

for spec in FINETUNED_SPECS:
    for epoch in spec['epochs']:
        checkpoint_path = CHECKPOINTS_ROOT / spec['tag'] / f'epoch_{epoch}'
        if not checkpoint_path.exists():
            print({'tag': spec['tag'], 'epoch': epoch, 'status': 'missing_checkpoint', 'path': str(checkpoint_path)})
            continue

        if spec['kind'] == 'florence':
            model, processor = load_florence(checkpoint_path)
        else:
            model, processor = load_qwen(spec['base_model_name'], checkpoint_path)

        mia_df = run_mia(model, processor, seen_small, unseen_small)
        auc_confidence = compute_auc_safe(mia_df['is_seen'], mia_df['confidence'])
        epoch_curve_rows.append({'tag': spec['tag'], 'epoch': epoch, 'auc_confidence': auc_confidence})
        experiment.log_metric(f"{spec['tag']}_auc", auc_confidence, step=epoch)
        print({'tag': spec['tag'], 'epoch': epoch, 'auc_confidence': auc_confidence})
        del model
        torch.cuda.empty_cache()

epoch_curve_df = pd.DataFrame(epoch_curve_rows).sort_values(['tag', 'epoch']).reset_index(drop=True)
epoch_curve_csv = MIA_OUTPUT_DIR / 'mia_epoch_curve.csv'
epoch_curve_df.to_csv(epoch_curve_csv, index=False)

fig, ax = plt.subplots(figsize=(8, 5))
for tag, group in epoch_curve_df.groupby('tag'):
    ax.plot(group['epoch'], group['auc_confidence'], marker='o', label=tag)
ax.set_xlabel('Epoch')
ax.set_ylabel('AUC-ROC (confidence)')
ax.set_title('MIA AUC by Epoch')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

experiment.log_table('mia_epoch_curve.csv', epoch_curve_df)
log_csv_asset(experiment, epoch_curve_csv)
experiment.log_figure('mia_epoch_curve', fig)
display(epoch_curve_df)
experiment.end()
plt.close(fig)

## 5. Saved Outputs

In [ ]:
saved_csvs = sorted(MIA_OUTPUT_DIR.glob('*.csv'))
saved_df = pd.DataFrame({'csv_path': [str(path) for path in saved_csvs]})
display(saved_df)
print({'output_dir': str(MIA_OUTPUT_DIR), 'csv_count': len(saved_csvs)})